In [12]:
import json
import requests
import pandas as pd

In [13]:


def query_dbpedia(sparql_query):
    endpoint = "https://dbpedia.data.dice-research.org/sparql"
    headers = {"Accept": "application/sparql-results+json"}
    params = {"query": sparql_query}
    try:
        response = requests.get(endpoint, headers=headers, params=params)
        response.raise_for_status()  # Raise an exception for HTTP errors
        return response.json()
    except requests.exceptions.RequestException as e:
        #print(f"Error querying DBpedia: {e}")
        return None

In [14]:
df = pd.read_csv("combined_unique_select_queries.csv")
print(len(df))



307553


In [18]:
df.tail()

,text
307548,PREFIX dbpr: <http://dbpedia.org/resource/>\n...
307549,PREFIX dbpr: <http://dbpedia.org/resource/>\n...
307550,PREFIX dbpr: <http://dbpedia.org/resource/>\n...
307551,PREFIX dbpr: <http://dbpedia.org/resource/>\n...
307552,PREFIX dbpr: <http://dbpedia.org/resource/>\n...


In [ ]:
counter = 0
failed = 0
successful_queries = []

for index, row in df.iterrows():
    sparql_query = row.get("text")

    if sparql_query:
        results = query_dbpedia(sparql_query)

       # if counter > 1 and counter % 100 == 0:
        #    print(counter)

        if results and "results" in results and results["results"]["bindings"]:
            successful_queries.append({
                "sparql_query": sparql_query,
                "results": results["results"]["bindings"]
            })
            counter += 1
        else:
            failed += 1


In [20]:
len(successful_queries)

217701

In [21]:
failed

89852

In [22]:
successful_queries[2100]

{'sparql_query': 'PREFIX  dbpr: <http://dbpedia.org/resource/>\n\nSELECT DISTINCT  (COUNT(?p) AS ?count)\nWHERE\n  {   { ?o  ?p  dbpr:Star_in_the_Night ;\n            ?p  dbpr:Executive_Decision\n      }\n    UNION\n      { ?o        ?p  dbpr:Star_in_the_Night .\n        dbpr:Executive_Decision\n                  ?p  ?o\n      }\n    UNION\n      { dbpr:Star_in_the_Night\n                  ?p  ?o .\n        ?o        ?p  dbpr:Executive_Decision\n      }\n    UNION\n      { dbpr:Star_in_the_Night\n                  ?p  ?o .\n        dbpr:Executive_Decision\n                  ?p  ?o\n      }\n  }\n',
 'results': [{'count': {'type': 'literal',
    'value': '34',
    'datatype': 'http://www.w3.org/2001/XMLSchema#integer'}}]}

In [23]:
import json

# Save to JSON file
with open("successful_queries.json", "w", encoding="utf-8") as f:
    json.dump(successful_queries, f, ensure_ascii=False, indent=2)


In [24]:
len(data)

217701

In [1]:
import json
import multiprocessing
from itertools import combinations
from collections import defaultdict
from tqdm import tqdm




In [2]:
# ---- Load Data ----
with open("successful_queries.json", "r") as f:
    data = json.load(f)


In [3]:
# ---- Function to extract result strings ----
def extract_result_strings(entry):
    values = set()
    for result in entry.get("results", []):
        for k, v in result.items():
            if isinstance(v, dict) and 'value' in v:
                values.add(f"{k}:{v['value']}")
    return values



In [4]:
# Preprocess all result sets once
preprocessed = [(i, entry['sparql_query'], extract_result_strings(entry)) for i, entry in enumerate(data)]




In [5]:
preprocessed

(0,
 'PREFIX  dbpr: <http://dbpedia.org/resource/>\n\nSELECT DISTINCT  (COUNT(?p) AS ?count)\nWHERE\n  {   { dbpr:Nederrijn\n                  ?p  dbpr:Bodrog\n      }\n    UNION\n      { dbpr:Bodrog  ?p  dbpr:Nederrijn }\n  }\n',
 {'count:0'})

In [6]:
# ---- Worker function for multiprocessing ----
def process_query(args):
    base_id, base_query, base_results = args
    local_similars = []

    for sim_id, sim_query, sim_results in preprocessed[base_id + 1:]:  # Only look ahead
        intersection = base_results & sim_results
        union = base_results | sim_results
        jaccard_score = len(intersection) / len(union) if union else 0

        if jaccard_score > 0.8:
            local_similars.append({
                "similar_query": sim_query.strip(),
                "jaccard_score": round(jaccard_score, 3),
                "similar_results": list(sim_results)
            })

            if len(local_similars) >= 5:
                break  # early stop

    return {
        "base_query": base_query.strip(),
        "base_results": list(base_results),
        "similar_queries": local_similars
    } if local_similars else None



In [ ]:
    args_list = [(i, q, r) for i, (idx, q, r) in enumerate(preprocessed)]

    results = []
    with multiprocessing.Pool(cpu_count) as pool:
        for result in tqdm(pool.imap_unordered(process_query, args_list), total=len(args_list)):
            if result:
                results.append(result)

    # ---- Write to JSON ----
    with open("jaccard_similarities_optimized.json", "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    print("✅ JSON file written: jaccard_similarities_optimized.json")

100%|█████████▉| 217700/217701 [5:49:54<41:44, 2504.37s/it]  

In [11]:
len(results)

185075

In [25]:
results[0]


{'base_query': 'PREFIX  dbpr: <http://dbpedia.org/resource/>\n\nSELECT DISTINCT  (COUNT(?p) AS ?count)\nWHERE\n  {   { ?o  ?p  dbpr:Copaifera_langsdorffii ;\n            ?p  dbpr:Hops\n      }\n    UNION\n      { ?o        ?p  dbpr:Copaifera_langsdorffii .\n        dbpr:Hops  ?p  ?o\n      }\n    UNION\n      { dbpr:Copaifera_langsdorffii\n                  ?p  ?o .\n        ?o        ?p  dbpr:Hops\n      }\n    UNION\n      { dbpr:Copaifera_langsdorffii\n                  ?p  ?o .\n        dbpr:Hops  ?p  ?o\n      }\n  }',
 'base_results': ['count:9'],
 'similar_queries': [{'similar_query': 'PREFIX  dbpr: <http://dbpedia.org/resource/>\n\nSELECT DISTINCT  (COUNT(?p) AS ?count)\nWHERE\n  {   { ?o  ?p  dbpr:Nederrijn ;\n            ?p  <http://dbpedia.org/resource/Laubach_(Wetter)>\n      }\n    UNION\n      { ?o        ?p  dbpr:Nederrijn .\n        <http://dbpedia.org/resource/Laubach_(Wetter)>\n                  ?p  ?o\n      }\n    UNION\n      { dbpr:Nederrijn\n                  ?p 